# 2.04 · MiBici model evaluation: rolling-origin backtest and the champion

**Phase 2 (model building)** following the [Cookiecutter Data Science](https://cookiecutter-data-science.drivendata.org/using-the-template/) convention: `PHASE.NOTEBOOK-initials-description`. It runs after `2.03`, which tuned the two finalists `lgbm_tweedie` and `hgb_poisson`.

- **Input:** the MLflow runs of `python -m team2_mlops2026.modeling.train backtest` and `... ablation`:
  - one parent run `backtest-<model>` per baseline and finalist, with one nested run per fold (`fold-2025-10` ... `fold-2026-09`), and the fold table, the error by hour, the decision metric, the out-of-sample forecasts, the permutation importance and the figures as artifacts under `backtest/`
  - one run `ablation-<model>-<target>-pca` / `-no-pca` per finalist and target, tagged `experiment=pca_ablation`
- **Output:** the champion of each target, with the numbers that justify it, for the Model Registry (`2.05`, #27).

**Why.** `2.02` and `2.03` scored the models on one split: trained until 2025-09 and tested on the next 12 months at once. The [ML Canvas](../references/ml-canvas.md) (§4, Impact simulation) asks for a rolling-origin backtest instead: train up to month *m*, forecast month *m + 1*, move one month on, as the monthly retraining would. Each of the 12 folds refits every model from scratch, the finalists with the params of their `tune-<model>-<target>` run. The success criterion is applied to the mean over the folds: **a peak-hour WAPE at least 10% lower than the best baseline, with no demand group worse.**

1. The backtest in MLflow and its lineage.
2. The mean and the spread over the 12 folds.
3. Success criterion and champion.
4. Error per test month: drift.
5. Error by hour of the day.
6. Decision metric: high-imbalance station-days.
7. Forecasts against actual values.
8. Permutation importance.
9. PCA ablation: does PCA help the model?

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os

os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"  # keep the outputs readable

import mlflow
import numpy as np
import pandas as pd

from team2_mlops2026 import plots
from team2_mlops2026.config import MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_URI
from team2_mlops2026.features import RUN_NAME as FEATURES_RUN
from team2_mlops2026.features import STATION, TARGETS, TIME
from team2_mlops2026.modeling.backtest import (
    ABLATION_PREFIX,
    ARTIFACT_DIR,
    BACKTEST_MODELS,
    BY_HOUR_FILE,
    FOLD,
    FORECAST,
    FORECASTS_FILE,
    IMBALANCE_FILE,
    NEW_STATIONS,
    load_backtests,
    rank_backtests,
    sample_stations,
)
from team2_mlops2026.modeling.evaluate import MIN_IMPROVEMENT, N_FOLDS, success_criterion
from team2_mlops2026.modeling.models import BASELINES, MODELS
from team2_mlops2026.modeling.tuning import FINALISTS
from team2_mlops2026.modeling.tuning import RUN_PREFIX as TUNE_PREFIX
from team2_mlops2026.station_profiles import RUN_NAME as PROFILES_RUN
from team2_mlops2026.tracking import input_digests, latest_run

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
plots.use_style()
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

## Parameters

`METRICS` are the ones the canvas fixes (WAPE, MAE and Poisson deviance), at the segments the success criterion reads: overall, at weekday peak hours, and the daily WAPE of the quietest stations (group 1) and of the stations opened since May 2026.

In [ ]:
METRICS = [
    "overall_wape",
    "peak_wape",
    "overall_mae",
    "peak_mae",
    "overall_poisson_deviance",
    "group_1_daily_wape",
    f"{NEW_STATIONS}_daily_wape",
]
GROUPS = [f"group_{i}" for i in range(1, 6)]
ABLATION_METRICS = [
    "overall_wape",
    "peak_wape",
    "overall_poisson_deviance",
    "group_1_daily_wape",
    "unseen_stations_daily_wape",
]


def artifact(run, name: str) -> str:
    """Local path of the backtest artifact ``name`` of ``run``."""
    return mlflow.artifacts.download_artifacts(
        run_id=run.info.run_id, artifact_path=f"{ARTIFACT_DIR}/{name}"
    )


def fold_reports(table: pd.DataFrame, target: str) -> dict[str, pd.DataFrame]:
    """The segment report of each fold, from a model's fold table."""
    rows = table[table["target"] == target]
    return {
        fold: r.drop(columns=[FOLD, "target"]).set_index("segment")
        for fold, r in rows.groupby(FOLD)
    }

## 1. The backtest in MLflow

The notebook reads the latest finished `backtest-<model>` run of each model, and its fold table (`backtest/folds.csv`: one row per fold, target and segment). Three checks:
- every parent run has its 12 nested fold runs, finished;
- its input digests are those of the latest `build-station-hour-features` and `fit-station-profiles` runs, so all models read the same data;
- each finalist took its params from the latest `tune-<model>-<target>` run, the one `2.03` analysed.

In [ ]:
backtests = load_backtests(BACKTEST_MODELS)
runs = {name: run for name, (run, _) in backtests.items()}
fold_tables = {name: table for name, (_, table) in backtests.items()}

upstream = {}
for run_name in (FEATURES_RUN, PROFILES_RUN):
    upstream |= input_digests(latest_run(run_name).info.run_id)

overview = {}
for name, run in runs.items():
    nested = mlflow.search_runs(
        experiment_names=[MLFLOW_EXPERIMENT_NAME],
        filter_string=f"tags.mlflow.parentRunId = '{run.info.run_id}'",
    )
    assert len(nested) == int(run.data.params["n_folds"]) == N_FOLDS
    assert (nested["status"] == "FINISHED").all()
    logged = input_digests(run.info.run_id)
    assert logged == {k: upstream[k] for k in logged}
    for target in TARGETS:
        source = run.data.tags[f"params_from_{target}"]
        if name in FINALISTS:
            assert source == latest_run(f"{TUNE_PREFIX}-{name}-{target}").info.run_id
    overview[name] = {
        "family": run.data.tags["model_family"],
        "parent run": run.info.run_id,
        "fold runs": len(nested),
        "test months": f"{run.data.params['first_test_month']} to {run.data.params['last_test_month']}",
        "params": "tuned" if name in FINALISTS else "rule",
        "fit minutes": round(sum(run.data.metrics[f"{t}_fit_seconds"] for t in TARGETS) / 60, 1),
    }
print(f"Experiment '{MLFLOW_EXPERIMENT_NAME}' ({MLFLOW_TRACKING_URI})")
pd.DataFrame(overview).T

**Result → action.** TODO

## 2. The mean and the spread over the 12 folds

Each parent run logs the mean of every fold metric as `<target>_<segment>_<metric>` and its standard deviation with a `_std` suffix. These are the columns to select in the MLflow UI: filter the runs with `attributes.run_name LIKE 'backtest-%'`, open **Compare**, and the bar charts show each model side by side. In the **Chart** view, the step metrics `fold_<target>_peak_wape` draw one line per model across the 12 folds.

In [ ]:
mean = pd.DataFrame(
    {
        (name, target): {m: run.data.metrics.get(f"{target}_{m}", np.nan) for m in METRICS}
        for name, run in runs.items()
        for target in TARGETS
    }
).T.rename_axis(["model", "target"])
std = pd.DataFrame(
    {
        (name, target): {m: run.data.metrics.get(f"{target}_{m}_std", np.nan) for m in METRICS}
        for name, run in runs.items()
        for target in TARGETS
    }
).T.rename_axis(["model", "target"])
order = pd.MultiIndex.from_product([TARGETS, BACKTEST_MODELS], names=["target", "model"])
(mean.round(3).astype(str) + " ± " + std.round(3).astype(str)).swaplevel().reindex(order)

In [ ]:
peak = mean["peak_wape"].unstack("target").loc[list(BACKTEST_MODELS), list(TARGETS)]
bar = {t: (1 - MIN_IMPROVEMENT) * peak.loc[list(BASELINES), t].min() for t in TARGETS}
print("Success bar (peak WAPE at most):", {t: round(v, 3) for t, v in bar.items()})
plots.plot_grouped_bars(
    peak, "Peak-hour WAPE, mean of the 12 test months (dashed: the success bar)", "WAPE", bar
);

**Result → action.** TODO

## 3. Success criterion and champion

`rank_backtests` applies `success_criterion` to the mean report of each finalist over the 12 folds, against the best baseline at peak hours, and ranks the finalists: the ones that pass first, then by peak-hour WAPE. The demand groups are compared on their daily WAPE (canvas §4). The first row of each target is the champion if it passed; the second is the challenger.

In [ ]:
rankings = rank_backtests(fold_tables)
pd.concat(rankings, names=["target"]).round(3)

The criterion on the mean could hide a bad month. The same check, fold by fold:

In [ ]:
monthly = {}
for target in TARGETS:
    baseline_folds = {b: fold_reports(fold_tables[b], target) for b in BASELINES}
    for name in FINALISTS:
        for fold, report in fold_reports(fold_tables[name], target).items():
            check = success_criterion(report, {b: baseline_folds[b][fold] for b in BASELINES})
            monthly[name, target, fold] = {
                "baseline": check.baseline,
                "improvement": check.peak_wape_improvement,
                "worse groups": ", ".join(check.worse_groups),
                "passed": check.passed,
            }
monthly = pd.DataFrame(monthly).T.rename_axis(["model", "target", "fold"])
monthly["improvement"] = monthly["improvement"].astype(float)
monthly.groupby(level=["model", "target"]).agg(
    folds_passed=("passed", "sum"),
    min_improvement=("improvement", "min"),
    mean_improvement=("improvement", "mean"),
    max_improvement=("improvement", "max"),
).round(3)

In [ ]:
failed = monthly[~monthly["passed"].astype(bool)]
failed.round(3) if len(failed) else print("Every finalist passes the criterion in every fold")

In [ ]:
champions = {}
for target, table in rankings.items():
    champion = table.index[0]
    row = table.loc[champion]
    assert row["passed"], f"No finalist passes the success criterion for {target}"
    champions[target] = champion
    challenger = table.index[1] if len(table) > 1 else row["baseline"]
    print(
        f"{target}: champion {champion}, peak WAPE {row['peak_wape']:.3f}, "
        f"{row['peak_wape_improvement']:.1%} below {row['baseline']} "
        f"({peak.loc[row['baseline'], target]:.3f}); groups worse: {row['worse_groups'] or 'none'}. "
        f"Challenger: {challenger}"
    )

**Result → action.** TODO

## 4. Error per test month: drift

The canvas expects two changes in the test year: the record demand of September 2026, and about 100 stations opened between May and August 2026 (the `new_stations` segment, scored per station-day). Each line is a model; the baselines are dashed.

In [ ]:
for target in TARGETS:
    by_month = pd.DataFrame(
        {
            name: table[(table["target"] == target) & (table["segment"] == "peak")].set_index(FOLD)[
                "wape"
            ]
            for name, table in fold_tables.items()
        }
    )
    plots.plot_lines(
        by_month,
        f"{target}: peak-hour WAPE per test month",
        "WAPE",
        "test month",
        muted=list(BASELINES),
    )

In [ ]:
forecasts = {name: pd.read_parquet(artifact(run, FORECASTS_FILE)) for name, run in runs.items()}
first = next(iter(forecasts.values()))
for frame in forecasts.values():  # every model forecast the same station-hours, in the same order
    assert frame[[STATION, TIME]].equals(first[[STATION, TIME]])

champion = champions["departures"]
drift = pd.DataFrame(
    {
        "departures (thousands)": first.groupby(FOLD)["departures"].sum() / 1e3,
        "stations": first.groupby(FOLD)[STATION].nunique(),
        "new stations": first[first[STATION].isin(
            first.loc[first[FOLD] >= "2026-05", STATION].unique()
        )].groupby(FOLD)[STATION].nunique(),
    }
)
for name in (champion, *BASELINES):
    rows = fold_tables[name]
    rows = rows[(rows["target"] == "departures") & (rows["segment"] == NEW_STATIONS)]
    drift[f"{name}: new stations daily WAPE"] = rows.set_index(FOLD)["daily_wape"]
drift.round(3)

**Result → action.** TODO

## 5. Error by hour of the day

WAPE of the out-of-sample forecasts of the 12 folds, per hour of the day (`backtest/error_by_hour.csv`).

In [ ]:
by_hour = {name: pd.read_csv(artifact(run, BY_HOUR_FILE)) for name, run in runs.items()}
for target in TARGETS:
    table = pd.DataFrame(
        {name: t[t["target"] == target].set_index("hour")["wape"] for name, t in by_hour.items()}
    )
    plots.plot_lines(
        table, f"{target}: WAPE by hour of the day", "WAPE", "hour of the day", muted=list(BASELINES)
    )

**Result → action.** TODO

## 6. Decision metric: high-imbalance station-days

Precision and recall in flagging the station-days whose absolute net flow exceeds half of the capacity (canvas §4), per fold. Each fold scores the departures and arrivals forecasts of the same model together.

In [ ]:
imbalance = pd.concat(
    {name: pd.read_csv(artifact(run, IMBALANCE_FILE), index_col=FOLD) for name, run in runs.items()},
    names=["model"],
)
imbalance.groupby(level="model")[
    ["imbalance_days", "flagged_days", "imbalance_precision", "imbalance_recall"]
].mean().reindex(BACKTEST_MODELS).round(3)

In [ ]:
for metric in ("imbalance_precision", "imbalance_recall"):
    plots.plot_lines(
        imbalance[metric].unstack("model")[list(BACKTEST_MODELS)],
        f"{metric.replace('_', ' ')} per test month",
        metric.split("_")[1],
        "test month",
        muted=list(BASELINES),
    )

**Result → action.** TODO

## 7. Forecasts against actual values

The first full week of the last test month, at three stations: the busiest, a median one and the busiest of the stations opened since May 2026. The champion against the best baseline, both refitted on the hours before September 2026.

In [ ]:
for target in TARGETS:
    champion = champions[target]
    baseline = rankings[target].loc[champion, "baseline"]
    last = first[FOLD].max()
    days = first.loc[first[FOLD] == last, TIME].dt.normalize()
    monday = days[days.dt.weekday == 0].min()
    week = (first[FOLD] == last) & first[TIME].between(monday, monday + pd.Timedelta(days=7), "left")
    new = first.loc[first[FOLD] >= "2026-05", STATION].unique()
    panels = {}
    for label, station in sample_stations(first, target, pd.Index(new)).items():
        rows = week & (first[STATION] == station)
        panels[f"station {station} ({label})"] = pd.DataFrame(
            {
                "actual": first.loc[rows, target].to_numpy(),
                champion: forecasts[champion].loc[rows, FORECAST.format(target)].to_numpy(),
                baseline: forecasts[baseline].loc[rows, FORECAST.format(target)].to_numpy(),
            },
            index=first.loc[rows, TIME],
        )
    plots.plot_station_forecasts(panels, f"Hourly {target} from {monday:%Y-%m-%d}", target)

In [ ]:
for target in TARGETS:
    champion = champions[target]
    baseline = rankings[target].loc[champion, "baseline"]
    plots.plot_residuals(
        pd.DataFrame(
            {
                name: forecasts[name][target] - forecasts[name][FORECAST.format(target)]
                for name in (champion, baseline)
            }
        ),
        f"{target}: residuals over the 12 test months",
    )

**Result → action.** TODO

## 8. Permutation importance

How much the Poisson deviance of the champion rises when one input column is shuffled, on 50,000 hours of the last test month, refitted on all the hours before it. The columns are the raw inputs: `station_id` covers its target encoding, and `pc1`-`pc5` are the PCA profile scores.

In [ ]:
importance = {
    target: pd.read_csv(artifact(runs[champions[target]], f"importance_{target}.csv"), index_col=0)
    for target in TARGETS
}
for target, table in importance.items():
    plots.plot_importance(table, f"{champions[target]}, {target}: permutation importance")
pd.DataFrame({t: table["mean"] for t, table in importance.items()}).sort_values(
    TARGETS[0], ascending=False
).round(4)

**Result → action.** TODO

## 9. PCA ablation: does PCA help the model?

The Phase 1 review asked whether the PCA station profiles (26 components for 90% of the variance in `1.05`; the models read the first five) really improve the model, or whether the original variables would do. Each finalist was scored on the test split of `2.02` (trained until 2025-09, tested on 2025-10 to 2026-09) with its tuned params, with and without the profile scores and their missing flag. The variant with the scores is the model the `tune-<model>-<target>` run already refitted on that split.

The original variables of the profile are 336 hour-of-week shares per station-week. Joined to 12.2 million station-hours they would need about 16 GB as float32, so the experiment compares the PCA summary with no profile at all.

In [ ]:
ablation = mlflow.search_runs(
    experiment_names=[MLFLOW_EXPERIMENT_NAME],
    filter_string=f"tags.experiment = 'pca_ablation' and status = 'FINISHED'",
    order_by=["start_time DESC"],
)
ablation = ablation.drop_duplicates("tags.mlflow.runName")
assert set(ablation["tags.mlflow.runName"]) == {
    f"{ABLATION_PREFIX}-{name}-{target}-{variant}"
    for name in FINALISTS
    for target in TARGETS
    for variant in ("pca", "no-pca")
}
ablation["variant"] = np.where(
    ablation["tags.mlflow.runName"].str.endswith("-no-pca"), "without PCA", "with PCA"
)
pca = ablation.set_index(["params.model", "tags.target", "variant"])[
    [f"metrics.{m}" for m in ABLATION_METRICS]
].rename(columns=lambda c: c.removeprefix("metrics."))
pca = pca.unstack("variant").swaplevel(axis=1).sort_index(axis=1)
for m in ABLATION_METRICS:
    pca[("change", m)] = pca[("without PCA", m)] / pca[("with PCA", m)] - 1
pca = pca.rename_axis(["model", "target"])
pca[["with PCA", "without PCA", "change"]].style.format("{:.4f}").format(
    "{:+.2%}", subset=[("change", m) for m in ABLATION_METRICS]
)

**Result → action.** TODO

## Findings and next steps

TODO